In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
pd.set_option("display.width", 200)

def find_project_root():
    path = Path.cwd().resolve()
    for candidate in [path, *path.parents]:
        if (candidate / "src" / "modeling.py").is_file():
            return candidate
    raise FileNotFoundError("Could not find src/modeling.py from " + str(path))

sys.path.insert(0, str(find_project_root() / "src"))
import importlib
import modeling as m
importlib.reload(m)  # pick up changes to modeling.py without restarting

dev = m.load("2017-2020")
y = dev[m.OUTCOME].to_numpy()

# Baseline 1: simple rule "refer everyone above age X" (no model at all)
age = dev["age"].to_numpy()
cutoff = m.threshold_for_sensitivity(y, age)
rule = m.at_threshold(y, age, cutoff)
print(f"Baseline 1: refer everyone aged >= {cutoff:.0f}")
print(f"  ROC-AUC (age alone): {m.roc_auc_score(y, age):.3f}")
for k in ["sensitivity", "specificity", "referred_pct", "screen_per_case"]:
    print(f"  {k:<16}{rule[k]:.3f}")

# Baselines 2-3 and the two models, all with the same cross-validation
candidates = {
    "Age only (logistic)": ["age"],
    "Age + sex": ["age", "female"],
    "Model A": m.MODEL_A,
    "Model B": m.MODEL_B,
}
rows = {}
for name, predictors in candidates.items():
    s = m.summarise(m.cross_validate(m.make_logistic, predictors, dev))["mean"]
    rows[name] = s[["roc_auc", "cal_slope", "obs_exp", "specificity",
                    "referred_pct", "screen_per_case"]]
print("\nComparison (cross-validated means, sensitivity fixed at 80%)")
print(pd.DataFrame(rows).T.round(3))

Baseline 1: refer everyone aged >= 43
  ROC-AUC (age alone): 0.733
  sensitivity     0.814
  specificity     0.526
  referred_pct    51.990
  screen_per_case 4.742

Comparison (cross-validated means, sensitivity fixed at 80%)
                     roc_auc  cal_slope  obs_exp  specificity  referred_pct  screen_per_case
Age only (logistic)    0.733      0.994    1.000        0.537        50.909            4.713
Age + sex              0.736      0.990    1.001        0.535        51.033            4.727
Model A                0.744      0.960    1.001        0.556        49.221            4.562
Model B                0.748      0.940    1.000        0.563        48.580            4.502
